In [2]:
!pip install ase


  Obtaining dependency information for ase from https://files.pythonhosted.org/packages/07/f5/007d993fcf3b051acb304d5402e0bd103fd20816b47dee9531bdbfb3aa0c/ase-3.25.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.0 MB 660.6 kB/s eta 0:00:05
   - -------------------------------------- 0.1/3.0 MB 1.2 MB/s eta 0:00:03
   --- ------------------------------------ 0.3/3.0 MB 2.0 MB/s eta 0:00:02
   -------- ------------------------------- 0.6/3.0 MB 3.6 MB/s eta 0:00:01
   ----------------- ---------------------- 1.3/3.0 MB 5.9 MB/s eta 0:00:01
   --------------------- ------------------ 1.6/3.0 MB 6.8 MB/s eta 0:00:01
   ----------------------------- ---------- 2.2/3.0 MB 7.2 MB/s eta 0:00:01
   ------------------------------- -------- 2.4/3.0 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 3.0/3.0 MB 7.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
comp-chem-utils 0.0.5 requires pymatgen>=2023.9.2, which is not installed.
comp-chem-utils 0.0.5 requires ruff==0.4.8, which is not installed.


In [1]:
import ase
from ase.io import read
print("✅ ASE installed successfully.")


✅ ASE installed successfully.


In [2]:
from ase.io import read, write
from ase.neighborlist import NeighborList
from ase.data import covalent_radii
import numpy as np
import os

# -----------------------------
# USER INPUT
input_file = "gen_0.extxyz"  # Replace with your file path
output_file = "clean_structure.xyz"   # Output file name
auto_delete_bad = False               # Set to True to delete bad files
# -----------------------------

# Load structure
atoms = read(input_file)
atoms.wrap()

# === 1. Check for overlapping atoms (< 0.7 Å) ===
positions = atoms.get_positions()
overlaps = []
for i in range(len(atoms)):
    for j in range(i + 1, len(atoms)):
        dist = np.linalg.norm(positions[i] - positions[j])
        if dist < 0.7:
            overlaps.append((i, j, dist))

# === 2. Check for disconnected fragments ===
cutoffs = [covalent_radii[atom.number] * 1.2 for atom in atoms]
nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
nl.update(atoms)

visited = set()
fragments = []

def dfs(atom_idx, frag):
    visited.add(atom_idx)
    frag.append(atom_idx)
    indices, offsets = nl.get_neighbors(atom_idx)
    for idx in indices:
        if idx not in visited:
            dfs(idx, frag)

for idx in range(len(atoms)):
    if idx not in visited:
        fragment = []
        dfs(idx, fragment)
        fragments.append(fragment)

# === Display Results ===
print(f"🧪 Structure checked: {input_file}")
print(f"📏 Total atoms: {len(atoms)}")

if overlaps:
    print(f"⚛️ Found {len(overlaps)} overlapping atom pairs (dist < 0.7 Å):")
    for i, j, d in overlaps:
        print(f"  ❌ Atoms {i} - {j} = {d:.3f} Å")
else:
    print("✅ No overlapping atoms found.")

print(f"🔗 Detected {len(fragments)} fragment(s)")
if len(fragments) > 1:
    print("⚠️ Structure is fragmented:")
    for i, frag in enumerate(fragments):
        print(f"  Fragment {i + 1}: {len(frag)} atoms")
else:
    print("✅ Structure is fully connected")

# === Final Decision ===
if overlaps or len(fragments) > 1:
    print("\n❌ Structure is INVALID.")
    if auto_delete_bad:
        print("🗑️ Deleting invalid file...")
        os.remove(input_file)
else:
    print("\n✅ Structure is VALID — saving cleaned version...")
    write(output_file, atoms)
    print(f"📦 Saved to {output_file}")


🧪 Structure checked: gen_0.extxyz
📏 Total atoms: 12
✅ No overlapping atoms found.
🔗 Detected 1 fragment(s)
✅ Structure is fully connected

✅ Structure is VALID — saving cleaned version...
📦 Saved to clean_structure.xyz
